# Fly Trajectory Prediction — Final Project

Runs the whole pipeline in Google Colab and produces every figure needed for
the project book and the presentation.

**Before you start:** `Runtime → Change runtime type → GPU`.
Step 3 onward is far faster on a GPU. Steps 1 and 2 do not use it.

### What this run does differently

The earlier run was a `quick` smoke test: 200,000 rows per file, which is only
about 33 flies, and just 4 of them in the test set. That was enough to prove
the code works and nothing more — four test flies is too small a sample to put
in front of a committee.

This run uses **all 326 flies**, and writes to a **new folder**
(`outputs_full/`) so the earlier results stay where they are and the two can
be compared.

It also predicts **50 ms ahead** rather than 17 ms, because the earlier
horizon sweep showed that is where the learned model's advantage over simple
physics is largest.

### What you get at the end

| File | What it shows |
|---|---|
| `pipeline_diagram.png` | the whole project on one page — opening slide |
| `model_diagram.png` | what is inside the predictor |
| `ksweep_silhouette.png` | how cluster quality varies with k, and the chosen k |
| `clusters_plot.png` | the trajectories inside each movement cluster |
| `epoch_progression.png` | **accuracy at epoch 0, 5, 10, … — proof training helps** |
| `learning_curve.png` | the same idea per epoch, on validation data |
| `per_fly_trajectories/` | **one figure per fly: model path, physics path, epochs** |
| `per_fly_summary.csv` | every fly's numbers in one table |
| `error_by_condition.png` | when the model helps: by speed and by turn rate |
| `error_by_cluster.png` | which movement types the model is good at |
| `horizon_efficiency.png` | **how far ahead it is worth predicting** |
| `prediction_examples.png` | individual moments, true vs. predicted |
| `prediction_results.csv` | the final error table |

### If the run gets interrupted

Step 1 writes its results after **every value of k**, and the horizon sweep
after **every horizon**. Everything goes straight to Drive. If Colab
disconnects, re-run the cell: it skips whatever is already done.

### Roughly how long

| Step | Time |
|---|---|
| 1 — clustering | ~50 min |
| 2 — TrajLearn prep + training | 20–40 min |
| 3 — the LSTM | 10–20 min on GPU |
| 4 — per-fly figures (326 of them) | 15–30 min |
| 5 — horizon sweep (5 more trainings) | 1–1.5 h |
| 0, 6 — the diagrams and the horizon slide | under a minute |

About 2.5–3.5 hours in total.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Locate the project folder

Even if Drive shows the folder under a localised name, the Colab mount path is
always `MyDrive`. This cell finds the project by itself and checks that
everything it needs is present.

In [ ]:
import os, glob

# If auto-detection fails, paste the full path here manually
PROJECT_DIR = None

if PROJECT_DIR is None:
    hits = glob.glob('/content/drive/MyDrive/**/fly_common.py', recursive=True)
    if len(hits) == 1:
        PROJECT_DIR = os.path.dirname(hits[0])
    elif len(hits) > 1:
        raise SystemExit('Found several copies, pick one manually:\n' +
                         '\n'.join(os.path.dirname(h) for h in hits))
    else:
        raise SystemExit('fly_common.py not found under MyDrive. '
                         'Make sure the project files are in Drive.')

print('Project folder:', PROJECT_DIR)

scripts = ['fly_common.py', '01_cluster_ksweep.py', '02_prepare_trajlearn.py',
           '03_continuous_model.py', '04_report_figures.py',
           '05_horizon_figure.py', '06_architecture_diagram.py']
missing = [s for s in scripts if not os.path.exists(os.path.join(PROJECT_DIR, s))]
if missing:
    raise SystemExit('Missing files: ' + ', '.join(missing))

csvs = [c for c in glob.glob(os.path.join(PROJECT_DIR, '**', '*.csv'), recursive=True)
        if 'outputs' not in c]
print(f'Found {len(csvs)} raw CSV files')
if not csvs:
    raise SystemExit('No CSV files found - check that the data folder is inside.')

## 3. Install packages

`torch`, `pandas` and `sklearn` ship with Colab. Only `tslearn` is missing.

In [ ]:
!pip install -q tslearn

In [ ]:
import torch, tslearn, sklearn
print('torch  ', torch.__version__, '| GPU:', torch.cuda.is_available())
print('tslearn', tslearn.__version__)
print('sklearn', sklearn.__version__)
if not torch.cuda.is_available():
    print('\nNO GPU. Steps 3-5 will still run, but many times more slowly.')
    print('Turn it on: Runtime -> Change runtime type -> GPU, then re-run.')

## 4. Settings for this run

| Mode | Rows per file | Time for step 1 | What it is for |
|---|---|---|---|
| `quick` | 200,000 (~33 flies) | ~10 min | checking the code runs at all |
| `medium` | **all 326 flies** | ~50 min | **the real run — use this** |
| `full` | all 326 flies | 2–2.5 h | same data, finer k sweep only |

`medium` already uses every fly. `full` differs only in how many segments the
k sweep samples, which changes the silhouette curve slightly and nothing else.

**The prediction horizon is set to 3 frames (50 ms).** The earlier sweep found
that is where the model beats simple physics by the widest margin — 81%,
against 62% at 17 ms and 33% at 500 ms.

In [ ]:
MODE = 'medium'          # 'quick' | 'medium' | 'full'

# A new folder, so the earlier quick-run results in outputs/ are left alone
OUTPUT_FOLDER = 'outputs_full'

PRESETS = {
    'quick':  dict(nrows='200000', k_min='2', k_max='6',  sweep='400'),
    'medium': dict(nrows='none',   k_min='2', k_max='15', sweep='400'),
    'full':   dict(nrows='none',   k_min='2', k_max='15', sweep='1200'),
}
if MODE not in PRESETS:
    raise SystemExit(f'MODE must be one of: {list(PRESETS)}')
p = PRESETS[MODE]

OUTPUT_DIR = os.path.join(PROJECT_DIR, OUTPUT_FOLDER)
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ['FLY_DATA_DIR']   = PROJECT_DIR
os.environ['FLY_OUTPUT_DIR'] = OUTPUT_DIR
os.environ['FLY_NROWS']      = p['nrows']
os.environ['FLY_K_MIN']      = p['k_min']
os.environ['FLY_K_MAX']      = p['k_max']
os.environ['FLY_SWEEP_MAX']  = p['sweep']

# --- the prediction task ---
os.environ['FLY_HORIZON'] = '3'    # frames ahead: 3 = 50 ms (the best horizon)
os.environ['FLY_WINDOW']  = '10'   # frames of history the model sees

# --- training ---
os.environ['FLY_MAX_EPOCHS']       = '40'
os.environ['FLY_PATIENCE']         = '6'
os.environ['FLY_CHECKPOINT_EVERY'] = '5'   # snapshot at 0, 5, 10, 15, ...

# --- per-fly figures ---
os.environ['FLY_ROLLOUT_FRAMES'] = '120'   # 2 s drawn per fly
os.environ['FLY_FIGURE_DPI']     = '110'

print(f'Mode: {MODE}')
print(f"  rows per file      : {p['nrows']}")
print(f"  k range            : {p['k_min']}..{p['k_max']}")
print(f"  segments in sweep  : {p['sweep']}")
print(f"  predicting ahead   : 3 frames = 50 ms")
print(f"  max epochs         : 40, snapshot every 5")
print(f'Outputs -> {OUTPUT_DIR}')

prev = os.path.join(OUTPUT_DIR, 'ksweep_silhouette.csv')
if os.path.exists(prev):
    import pandas as pd
    print(f"\nA previous sweep exists here (k = "
          f"{sorted(pd.read_csv(prev)['k'].astype(int))}).")
    print('Step 1 will skip those. If you changed MODE, run the next cell.')

In [ ]:
# Run this cell ONLY if you changed MODE and want a clean sweep
# os.remove(os.path.join(OUTPUT_DIR, 'ksweep_silhouette.csv'))

In [ ]:
import subprocess, sys, time
from IPython.display import Image, display

def run_step(script, env_overrides=None):
    """Run a script inside PROJECT_DIR, streaming its output live."""
    env = os.environ.copy()
    if env_overrides:
        env.update({k: str(v) for k, v in env_overrides.items()})
    print(f'{"="*70}\n{script}\n{"="*70}', flush=True)
    t0 = time.time()
    proc = subprocess.Popen([sys.executable, '-u', script],
                            cwd=PROJECT_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print(f'\n[exit code {proc.returncode}, {(time.time()-t0)/60:.1f} min]')
    if proc.returncode != 0:
        raise RuntimeError(f'{script} failed - see the error above.')

def show(*names, folder=None):
    base = folder or OUTPUT_DIR
    for n in names:
        path = os.path.join(base, n)
        if os.path.exists(path):
            print(n)
            display(Image(path))
        else:
            print('(not created:', n + ')')

## Step 0 — the diagrams (instant)

These need no data, so run them now and you have your opening slides while
everything else is still computing.

In [ ]:
run_step('06_architecture_diagram.py')
show('pipeline_diagram.png', 'model_diagram.png')

## Step 1 — Clustering: how many kinds of movement are there?

The slowest step: every value of k is a full clustering run.

**If it gets interrupted, just run this cell again.** Results are saved after
each k and already-computed values are skipped.

In [ ]:
run_step('01_cluster_ksweep.py')

In [ ]:
show('ksweep_silhouette.png', 'clusters_plot.png')

**How to read this.** The first plot is silhouette against k; the peak is the
number of movement types. Do not expect a value near 1 — fly movement is a
continuum rather than sharply separated categories, so a low but positive peak
is the honest finding, and saying so is better than overselling the clusters.

The second plot shows the trajectories inside each cluster. A good cluster has
lines that resemble each other. Curving up = turning left, down = right.

## Step 2 — TrajLearn (the discrete-cell model)

Check that the line `Split verified` appears. If it does not, stop: it means a
fly leaked between train and test and the results are not valid.

In [ ]:
run_step('02_prepare_trajlearn.py')

In [ ]:
import shutil

TRAJ_DIR = '/content/Trajectory-prediction'
!rm -rf $TRAJ_DIR
!git clone -q https://github.com/amir-ni/Trajectory-prediction $TRAJ_DIR
!pip install -q h3

src = os.path.join(OUTPUT_DIR, 'trajlearn')
shutil.copytree(os.path.join(src, 'flies'),
                os.path.join(TRAJ_DIR, 'data', 'flies'), dirs_exist_ok=True)
shutil.copy(os.path.join(src, 'configs.yaml'), TRAJ_DIR)

cfg_path = os.path.join(TRAJ_DIR, 'configs.yaml')
cfg = open(cfg_path).read()
if not torch.cuda.is_available():
    cfg = cfg.replace('device: cuda', 'device: cpu')
    open(cfg_path, 'w').write(cfg)
    print('Set to cpu\n')
print(cfg)

In [ ]:
%cd /content/Trajectory-prediction
!python main.py configs.yaml
!python main.py configs.yaml --test

## Step 3 — The main model

This is the step that answers the project's goal: where the fly will be 50 ms
from now, in millimetres.

It snapshots the model at epoch 0 (untrained), 5, 10, 15… and scores every
snapshot on the same held-out test flies. That is what turns "the model gets
better as it trains" from a claim into a measurement.

In [ ]:
run_step('03_continuous_model.py')

In [ ]:
import pandas as pd
print('Final test-set results:')
display(pd.read_csv(os.path.join(OUTPUT_DIR, 'prediction_results.csv')))
print('\nScore of each saved snapshot, on the test set:')
display(pd.read_csv(os.path.join(OUTPUT_DIR, 'epoch_checkpoints.csv')))
show('epoch_progression.png', 'learning_curve.png',
     'error_by_condition.png', 'prediction_examples.png',
     'prediction_error_plot.png')

### Which evidence to trust

**`epoch_progression.png` is the headline figure.** Every point is the same
unseen test flies, scored with a different amount of training behind it. The
only thing changing along the x-axis is how long the model trained, so the
downward slope is the improvement — measured, not asserted.

**`learning_curve.png`** shows the same idea per epoch on validation data, the
data training was allowed to look at. Use it to say *when* the model overtook
physics and when it stopped improving.

**Do not draw conclusions from `prediction_examples.png` alone.** Those 12
panels are illustrations chosen to span easy to hard. They show what the
numbers *mean*; 12 moments out of hundreds of thousands prove nothing.

A defensible claim sounds like: *"over N test moments from M held-out flies,
the model's median error was X mm against the baseline's Y mm, and it was
closer on Z% of moments"* — with the example panels as illustration
afterwards, never as the argument itself.

## Step 4 — One figure per fly

For every fly: the path it really took, the path the model predicts, the path
simple physics predicts, and the model's path at 3–4 different points during
training — plus the error moment by moment along that stretch.

The figures land in `outputs_full/per_fly_trajectories/`, one PNG per fly,
named so that test flies are obvious (`fly_0042_test.png`). **Judge the model
on the test flies** — a good-looking prediction on a fly it trained on proves
nothing.

This writes a few hundred files and takes 15–30 minutes. To try it on a
handful of flies first, uncomment the `FLY_MAX_FIGURES` line.

In [ ]:
# os.environ['FLY_MAX_FIGURES'] = '10'   # uncomment to test on 10 flies first
run_step('04_report_figures.py')

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = pd.read_csv(os.path.join(OUTPUT_DIR, 'per_fly_summary.csv'))
test = summary[summary['split'] == 'test']
print(f'{len(summary)} flies drawn, {len(test)} of them test flies.\n')

beat = test['onestep_median_model_mm'] < test['onestep_median_physics_mm']
print(f'The model is more accurate than physics on '
      f'{100*beat.mean():.0f}% of the test flies.')
print(f"Median per-fly error: {test['onestep_median_model_mm'].median():.4f} mm "
      f"(model) vs {test['onestep_median_physics_mm'].median():.4f} mm (physics)")
display(test.head(10))

# the clearest test fly and the hardest one
gap = test['onestep_median_physics_mm'] - test['onestep_median_model_mm']
fig_dir = os.path.join(OUTPUT_DIR, 'per_fly_trajectories')
for label, idx in [('BEST case', gap.idxmax()), ('WORST case', gap.idxmin())]:
    fly = int(test.loc[idx, 'fly'])
    path = os.path.join(fig_dir, f'fly_{fly:04d}_test.png')
    print(f'\n{label}: fly {fly}')
    if os.path.exists(path):
        display(Image(path))

In [ ]:
show('error_by_cluster.png')

**About the per-fly figures.** Panels (a)–(c) let each method run *free*: it
gets the first few frames and then predicts from its own predictions, with
nothing to correct it. Errors compound, so both paths drift — that is what
free-running prediction genuinely looks like, and it is the panel that shows
the difference at a glance.

Panel (d) is the fair comparison, and the one the reported accuracy is based
on: at every point on the *real* path, predict just 50 ms ahead. Errors cannot
accumulate there.

Quote panel (d)'s numbers. Show panels (a)–(c) to make them concrete.

## Step 5 — How far ahead is it worth predicting?

Trains the model again at several horizons and collects the results. Each
horizon writes to its own subfolder, so **the main 50 ms results from step 3
are not overwritten**.

This is the longest remaining step — five more trainings, roughly an hour on a
GPU. Results are saved after each horizon, so an interruption only costs the
one in progress, and re-running skips the horizons already done.

If time is short, cut `HORIZONS` down to `[1, 3, 15]` — three points still
show the peak.

In [ ]:
import pandas as pd

HORIZONS = [1, 3, 6, 15, 30]        # frames: 17 ms .. 500 ms

sweep_csv = os.path.join(OUTPUT_DIR, 'horizon_sweep.csv')
done, rows = set(), []
if os.path.exists(sweep_csv):
    prev = pd.read_csv(sweep_csv)
    rows = [prev]
    done = set(prev['horizon_frames'].astype(int))
    print(f'Resuming: horizons {sorted(done)} already done.')

for h in HORIZONS:
    if h in done:
        print(f'\n### horizon {h} frames - already done, skipping ###')
        continue
    sub = os.path.join(OUTPUT_DIR, 'horizon_runs', f'h{h:02d}')
    os.makedirs(sub, exist_ok=True)
    print(f'\n### horizon {h} frames ({h/60*1000:.0f} ms) -> {sub} ###')
    run_step('03_continuous_model.py',
             env_overrides={'FLY_HORIZON': h, 'FLY_OUTPUT_DIR': sub})
    r = pd.read_csv(os.path.join(sub, 'prediction_results.csv'))
    r['horizon_frames'] = h
    r['horizon_ms'] = h / 60 * 1000
    rows.append(r)
    # save after every horizon so an interruption keeps what already ran
    pd.concat(rows, ignore_index=True).to_csv(sweep_csv, index=False)
    print(f'Saved {sweep_csv}')

print('\nHorizon sweep complete.')
display(pd.read_csv(sweep_csv).pivot_table(
    index='horizon_ms', columns='model', values='median_error_mm'))

## Step 6 — The horizon slide

In [ ]:
run_step('05_horizon_figure.py')
show('horizon_efficiency.png')

**What this slide says.** Predicting further ahead is harder for everyone, so
both curves rise. But a straight-line assumption decays much faster than a
learned one, because over a longer window the fly has time to turn. The
model's *advantage* therefore rises, peaks, and falls again.

The peak is a real design result: it answers "how far ahead should this system
look?", and it did not have to come out anywhere in particular.

## What to put where

Everything below is in `outputs_full/`.

**For the presentation**, in order:

1. `pipeline_diagram.png` — what the project is
2. `model_diagram.png` — how the predictor works
3. `ksweep_silhouette.png` + `clusters_plot.png` — the clustering result
4. `epoch_progression.png` — training measurably helps
5. one or two figures from `per_fly_trajectories/` — what that means for a
   real fly
6. `horizon_efficiency.png` — choosing the horizon
7. `error_by_condition.png` or `error_by_cluster.png` — when the model helps

**For the book**, the same, plus `prediction_results.csv`,
`epoch_checkpoints.csv`, `per_fly_summary.csv` and `horizon_efficiency.csv`
as tables, and `prediction_error_plot.png` for the error distributions.

Run the cell below for a paste-ready version of the sentence to build the
results chapter around.

In [ ]:
import json, pandas as pd

res = pd.read_csv(os.path.join(OUTPUT_DIR, 'prediction_results.csv'))
ck = pd.read_csv(os.path.join(OUTPUT_DIR, 'epoch_checkpoints.csv'))
split = json.load(open(os.path.join(OUTPUT_DIR, 'fly_split.json'),
                       encoding='utf-8'))

m = res[res['model'] == 'LSTM (learned)'].iloc[0]
b = res[res['model'] == 'Constant velocity (baseline)'].iloc[0]
final, first = ck.iloc[-1], ck.iloc[0]

print('Paste-ready summary:\n')
print(f"Over held-out data from {len(split['test'])} flies the model had never "
      f"seen, the LSTM's median\nerror 50 ms ahead was "
      f"{m['median_error_mm']:.4f} mm, against {b['median_error_mm']:.4f} mm "
      f"for a constant-velocity\nbaseline - a {m['vs_baseline_%']:.1f}% "
      f"improvement - and it was the closer of the two on\n"
      f"{100*final['win_rate']:.1f}% of individual moments.")
print(f"\nBefore any training, the same architecture scored "
      f"{first['median_error_mm']:.4f} mm,\nso training accounts for a "
      f"{100*(first['median_error_mm']-final['median_error_mm'])/first['median_error_mm']:.0f}% "
      f"reduction in error.")
print(f"\nFlies: {len(split['train'])} train / {len(split['val'])} validation / "
      f"{len(split['test'])} test, split so that no fly appears in more than one.")